# 04c: TabPFN vs All Baselines - Comprehensive Comparison

**Purpose:** Statistical comparison of TabPFN (zero-shot & fine-tuned) against all baseline models

**Dataset:** COMPAS test set predictions from all 6 models

**Date:** 2025-11-08

---

## Overview

### Models Compared (6 total)
1. Logistic Regression (tuned)
2. XGBoost (tuned)
3. LightGBM (tuned)
4. CatBoost (tuned)
5. TabPFN Zero-Shot
6. TabPFN Fine-Tuned

### Statistical Framework
- **Primary test**: DeLong for AUROC comparisons (all pairwise)
- **Corrections**: Holm step-down (15 comparisons)
- **Effect sizes**: NNE, Cohen's d
- **Alpha**: 0.05 (family-wise error rate)

### Runtime: 2-3 minutes
---

In [ ]:
# Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score, roc_curve

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

from statistics_utils.hypothesis_tests import delong_test
from statistics_utils.multiple_comparisons import holm_correction
from statistics_utils.effect_sizes import number_needed_to_evaluate

PROCESSED_DIR = project_root / 'data' / 'processed'
PREDICTIONS_DIR = project_root / 'results' / 'predictions'
METRICS_DIR = project_root / 'results' / 'metrics'
TABLES_DIR = project_root / 'results' / 'tables'
FIGURES_DIR = project_root / 'results' / 'figures' / 'tabpfn'

for d in [TABLES_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
print('✓ Setup complete')

## 1. Load All Model Predictions

In [ ]:
# Load ground truth\ny_test = pd.read_parquet(PROCESSED_DIR / 'compas_y_test.parquet')['two_year_recid']\n\n# All models\nmodels = ['logistic_regression', 'xgboost', 'lightgbm', 'catboost', 'tabpfn_zeroshot', 'tabpfn_finetuned']\npredictions = {}\nmetrics_summary = []\n\nfor model in models:\n    pred_df = pd.read_parquet(PREDICTIONS_DIR / f'{model}_predictions.parquet')\n    test_df = pred_df[pred_df['split'] == 'test'].reset_index(drop=True)\n    predictions[model] = {'y_proba': test_df['y_proba'].values, 'y_pred': test_df['y_pred'].values}\n    \n    with open(METRICS_DIR / f'{model}_metrics.json', 'r') as f:\n        metrics = json.load(f)\n    metrics_summary.append({'model': model.replace('_', ' ').title(), 'auroc': metrics['test']['auroc'],\n                            'auprc': metrics['test']['auprc'], 'f1': metrics['test']['f1'],\n                            'brier': metrics['test']['brier_score']})\n\nmetrics_df = pd.DataFrame(metrics_summary).sort_values('auroc', ascending=False)\nprint('All Model Performance (Test Set):\n' + '='*60)\nprint(metrics_df.to_string(index=False))

## 2. Pairwise Statistical Comparisons

In [ ]:
# All pairwise DeLong tests\ndelong_results = []\nfor i, model1 in enumerate(models):\n    for model2 in models[i+1:]:\n        result = delong_test(y_test, predictions[model1]['y_proba'], predictions[model2]['y_proba'])\n        delong_results.append({'model1': model1.replace('_', ' ').title(), 'model2': model2.replace('_', ' ').title(),\n                              'auroc1': result['auroc1'], 'auroc2': result['auroc2'],\n                              'difference': result['auroc1'] - result['auroc2'],\n                              'p_value': result['p_value']})\n\ndelong_df = pd.DataFrame(delong_results)\n\n# Holm correction\np_values = delong_df['p_value'].values\nholm_result = holm_correction(p_values, alpha=0.05)\ndelong_df['p_holm'] = holm_result['adjusted_p_values']\ndelong_df['significant'] = holm_result['rejected']\n\nprint(f'\nPairwise DeLong Tests ({len(delong_df)} comparisons):\n' + '='*80)\nprint(delong_df[['model1', 'model2', 'difference', 'p_value', 'p_holm', 'significant']].to_string(index=False))\nprint(f'\nSignificant differences (Holm-corrected): {holm_result["num_rejected"]} of {len(delong_df)}')

## 3. Model Rankings

In [ ]:
# Add rankings\nfor metric in ['auroc', 'auprc', 'f1']:\n    metrics_df[f'{metric}_rank'] = metrics_df[metric].rank(ascending=False).astype(int)\n\nmetrics_df['brier_rank'] = metrics_df['brier'].rank(ascending=True).astype(int)  # Lower is better\n\nprint('\nModel Rankings:\n' + '='*80)\nprint(metrics_df.to_string(index=False))\n\nmetrics_df.to_csv(TABLES_DIR / 'comprehensive_model_comparison.csv', index=False)\nprint('\n✓ Saved comprehensive comparison')

## 4. ROC Curves (All Models)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))\ncolors = ['blue', 'orange', 'green', 'red', 'purple', 'brown']\nlinestyles = ['-', '-', '-', '-', '--', ':']\n\nfor (model, color, ls) in zip(models, colors, linestyles):\n    fpr, tpr, _ = roc_curve(y_test, predictions[model]['y_proba'])\n    auroc = metrics_df[metrics_df['model'] == model.replace('_', ' ').title()]['auroc'].values[0]\n    ax.plot(fpr, tpr, label=f\"{model.replace('_', ' ').title()} ({auroc:.3f})\", color=color, linestyle=ls, linewidth=2)\n\nax.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=1)\nax.set_xlabel('False Positive Rate', fontsize=12)\nax.set_ylabel('True Positive Rate', fontsize=12)\nax.set_title('ROC Curves: All Models', fontweight='bold', fontsize=14)\nax.legend(fontsize=9, loc='lower right')\nax.grid(alpha=0.3)\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'all_models_roc_comparison.png', dpi=300, bbox_inches='tight')\nplt.show()\nprint('✓ Saved ROC comparison')

## 5. Performance Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))\nmetrics_plot = [('auroc', 'AUROC'), ('auprc', 'AUPRC'), ('f1', 'F1 Score'), ('brier', 'Brier Score')]\n\nfor ax, (metric, title) in zip(axes.flat, metrics_plot):\n    x = np.arange(len(models))\n    values = [metrics_df[metrics_df['model'] == m.replace('_', ' ').title()][metric].values[0] for m in models]\n    bars = ax.bar(x, values, color=colors, alpha=0.7)\n    ax.set_xticks(x)\n    ax.set_xticklabels([m.replace('_', ' ').title() for m in models], rotation=45, ha='right', fontsize=9)\n    ax.set_ylabel(title, fontsize=11)\n    ax.set_title(f'{title} Comparison', fontweight='bold', fontsize=12)\n    ax.grid(axis='y', alpha=0.3)\n    for bar in bars:\n        height = bar.get_height()\n        ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.3f}', ha='center', va='bottom', fontsize=8)\n\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'all_models_metrics_bars.png', dpi=300, bbox_inches='tight')\nplt.show()\nprint('✓ Saved metrics comparison')

## Summary

**Comprehensive Model Comparison Complete:**
- ✓ 6 models compared statistically
- ✓ 15 pairwise DeLong tests with Holm correction
- ✓ Model rankings by all metrics
- ✓ ROC curves visualization
- ✓ Publication-ready tables and figures

**Key Findings:**
- Best AUROC: [Model name]
- TabPFN zero-shot rank: [Rank]
- TabPFN fine-tuned rank: [Rank]
- Significant differences: [Count]

**Next:** 04d_tabpfn_sensitivity.ipynb (Robustness analysis)